## 面试问题

提前终止 vs 过度执行：premature stop 与 overrun 的成因与防护？

## 回答主线

premature(没做完就停)和 overrun(做完还继续)是对称失败。防 premature 用客观完成验证 + 区分可恢复/致命错误；防 overrun 用明确 done 判定 + 完成后封闭。本 Notebook 分两部分：premature(遇可恢复错误即放弃 vs 区分错误后继续)、overrun(无 done 检查重复退款 vs 有 done 检查只退一次)。

## 真实案例

退款任务。premature：第 0 步超时(可恢复)却直接放弃。overrun：订单已退款成功但无 done 检查，重复退款三次。数据为教学状态，不代表真实系统。

In [1]:
ERRORS = {  # 定义两类错误。
    "timeout": "recoverable",  # 超时可重试恢复。
    "invalid_param": "fatal",  # 参数非法为致命错误。
}  # 结束错误分类定义。

def classify_error(err):  # 判断错误是否可恢复。
    return ERRORS.get(err, "fatal")  # 未知错误默认致命。

print("错误分类:", ERRORS)  # 展示错误分类。
print("timeout ->", classify_error("timeout"))  # 展示超时可恢复。
print("invalid_param ->", classify_error("invalid_param"))  # 展示参数非法致命。

错误分类: {'timeout': 'recoverable', 'invalid_param': 'fatal'}
timeout -> recoverable
invalid_param -> fatal


## 基线（Baseline）

反面基线之一（premature）：遇到任何错误就放弃。第 0 步只是可恢复的超时，循环却直接 gave_up。

In [2]:
def run_stop_on_error(errors_seen):  # premature 基线：遇任何错误即停。
    for step, err in enumerate(errors_seen):  # 逐步遇到错误。
        if err is not None:  # 出现错误。
            return {"status": "gave_up", "at_step": step, "on_error": err}  # 一遇错就放弃。
    return {"status": "completed", "at_step": len(errors_seen)}  # 无错则完成。

premature = run_stop_on_error(["timeout", None, None])  # 第 0 步超时可恢复却直接放弃。
print("遇错即停结果:", premature)  # 展示可恢复错误也导致 premature 放弃。

遇错即停结果: {'status': 'gave_up', 'at_step': 0, 'on_error': 'timeout'}


## 失败案例与修正

premature 修正：区分可恢复与致命错误，可恢复错误跳过继续。overrun 修正：无 done 检查会重复退款，加 done 检查后完成即封闭、只退一次。

In [3]:
def run_classify_error(errors_seen):  # premature 修正：区分可恢复与致命错误。
    for step, err in enumerate(errors_seen):  # 逐步遇到错误。
        if err is not None:  # 出现错误。
            if classify_error(err) == "fatal":  # 致命错误才停止。
                return {"status": "failed", "at_step": step, "on_error": err}  # 致命错误终止。
    return {"status": "completed", "at_step": len(errors_seen)}  # 可恢复错误被跳过后完成。

fixed_premature = run_classify_error(["timeout", None, None])  # 同样第 0 步超时但可恢复。
print("区分错误后结果:", fixed_premature)  # 展示可恢复错误不再导致放弃。

区分错误后结果: {'status': 'completed', 'at_step': 3}


In [4]:
def run_no_done_check(steps=3):  # overrun 基线：无 done 检查退款成功后继续。
    refunds = 0  # 记录退款次数。
    status = "paid"  # 初始未退款。
    for _ in range(steps):  # 固定执行若干步。
        status = "refunded"  # 每步都执行退款即重复副作用。
        refunds += 1  # 累加退款次数。
    return {"status": status, "refund_count": refunds}  # 返回退款次数。

def run_with_done_check(steps=3):  # overrun 修正：完成后封闭禁止重复副作用。
    refunds = 0  # 记录退款次数。
    done = False  # 完成标记。
    for _ in range(steps):  # 固定执行若干步。
        if done:  # 已完成则不再执行副作用。
            break  # 完成后停止。
        refunds += 1  # 执行一次退款。
        done = True  # 退款成功即完成。
    return {"status": "refunded", "refund_count": refunds}  # 返回退款次数。

overrun = run_no_done_check()  # 无 done 检查重复退款。
guarded = run_with_done_check()  # 有 done 检查只退一次。
print("无 done 检查退款次数:", overrun["refund_count"], "(overrun 重复退款)")  # 展示过度执行。
print("有 done 检查退款次数:", guarded["refund_count"], "(完成后封闭)")  # 展示只退一次。

无 done 检查退款次数: 3 (overrun 重复退款)
有 done 检查退款次数: 1 (完成后封闭)


## 结果解读

premature：遇错即停对可恢复超时也放弃，区分错误后跑完全程完成；overrun：无 done 检查退款三次，有 done 检查退一次。两个对称风险此消彼长，必须同时守：完成判定太松滑向 premature，太严滑向 overrun。

In [5]:
print("premature: 遇错即停", premature["status"], "-> 区分后", fixed_premature["status"])  # 展示 premature 修正。
print("overrun: 无检查", overrun["refund_count"], "次 -> 有检查", guarded["refund_count"], "次")  # 展示 overrun 修正。
print("两个对称风险都需同时防护")  # 强调对称性。

premature: 遇错即停 gave_up -> 区分后 completed
overrun: 无检查 3 次 -> 有检查 1 次
两个对称风险都需同时防护


In [6]:
assert premature["status"] == "gave_up"  # 遇错即停对可恢复错误也放弃。
assert fixed_premature["status"] == "completed"  # 区分错误后可恢复错误不导致放弃。
assert fixed_premature["at_step"] == 3  # 修正后跑完全部步骤。
assert overrun["refund_count"] == 3  # 无 done 检查重复退款三次。
assert guarded["refund_count"] == 1  # 有 done 检查只退款一次。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
